In [1]:
import importlib.util, subprocess, sys

required = {"kiwipiepy": "kiwipiepy", "sklearn": "scikit-learn", "pandas": "pandas"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import platform
import numpy as np
import pandas as pd
import sklearn
import kiwipiepy

print("Python", platform.python_version())
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("kiwipiepy", kiwipiepy.__version__)

Python 3.12.14
numpy 2.3.5
pandas 3.0.1
scikit-learn 1.9.1
kiwipiepy 0.23.2


# 2주차 과제: 벡터화·검색·평활화·형태소 분석

말뭉치의 각 줄은 `감성 라벨\t리뷰` 형식이다. 리뷰 한 줄을 문서 하나로 정의했다. Count, TF-IDF, BM25, n-gram 실험에서 아래의 같은 정규식 분석기를 사용했다. 형태소 오류 실험만 Kiwi를 사용했다.

In [2]:
from pathlib import Path
import urllib.request

CORPUS_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt"
CORPUS_PATH = "/content/corpus.txt"

corpus_path = Path(CORPUS_PATH)
corpus_path.parent.mkdir(parents=True, exist_ok=True)
if not corpus_path.exists():
    urllib.request.urlretrieve(CORPUS_URL, CORPUS_PATH)

lines = corpus_path.read_text(encoding="utf-8").splitlines()
labels, docs = [], []
for line in lines:
    label, text = line.split("\t", 1)
    labels.append(label)
    docs.append(text.strip())

TOKEN_PATTERN = r"(?u)\b[가-힣A-Za-z0-9]+\b"
print("파일:", CORPUS_PATH)
print(f"문서 수: {len(docs):,}")
print("첫 문서:", docs[0])

파일: /content/corpus.txt
문서 수: 100,000
첫 문서: 노래가 너무 적음


## 1. Count와 TF-IDF 상위어

Count는 단어의 출현 횟수를 그대로 세고, TF-IDF는 한 문서 안의 빈도에 전체 문서에서의 희귀성을 곱한다. 수업 자료와 맞추기 위해 `sublinear_tf=True`와 scikit-learn 기본 idf 스무딩을 사용했다.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(token_pattern=TOKEN_PATTERN, lowercase=False)
tfidf_vec = TfidfVectorizer(
    token_pattern=TOKEN_PATTERN,
    lowercase=False,
    sublinear_tf=True,
    smooth_idf=True,
    norm="l2",
)
X_count = count_vec.fit_transform(docs)
X_tfidf = tfidf_vec.fit_transform(docs)
vocab = count_vec.get_feature_names_out()
df = np.asarray((X_count > 0).sum(axis=0)).ravel()

print("행렬 크기:", X_count.shape)
print("0이 아닌 칸 비율:", f"{X_count.nnz / (X_count.shape[0] * X_count.shape[1]):.4%}")

행렬 크기: (100000, 240500)
0이 아닌 칸 비율: 0.0043%


In [4]:
selected = [79110, 19399, 97655]  # 0부터 세는 문서 인덱스

def top_indices(row, terms, n=5):
    values = row.toarray().ravel()
    return np.lexsort((terms, -values))[:n]

tfidf_rows = []
for doc_idx in selected:
    top_c = top_indices(X_count[doc_idx], vocab)
    top_t = top_indices(X_tfidf[doc_idx], vocab)
    for rank, (ci, ti) in enumerate(zip(top_c, top_t), 1):
        tfidf_rows.append({
            "문서": f"D{doc_idx + 1}",
            "순위": rank,
            "Count 단어": vocab[ci],
            "Count": int(X_count[doc_idx, ci]),
            "TF-IDF 단어": vocab[ti],
            "TF-IDF": round(float(X_tfidf[doc_idx, ti]), 4),
            "df": int(df[ti]),
            "idf": round(float(tfidf_vec.idf_[ti]), 4),
        })

for doc_idx in selected:
    print(f"D{doc_idx + 1}: {docs[doc_idx]}")
display(pd.DataFrame(tfidf_rows))

D79111: 연출 ★★★★★ 10/10 스토리 ★★★★★ 10/10 게임성 ★★★★ 8/10 그래픽 ★★★★★ 10/10 음악 ★★★★★ 10/10 평점 ★★★★★ 10/10 [인생작] 바이오쇼크, 게임을 넘어선 하나의 예술작품이 되다
D19400: 흔한 탑다운 슈팅 게임 장점 ⭕흔한 그래픽 ⭕흔한 브금 ⭕흔한 단순 조작 ⭕흔한 학살잼 단점 ❌흔한 안한글 ❌흔한 좀비학살 ❌흔한 노잼 ❌흔한 좀비슈터 아류작 🏷️ 좀비슈터가 학살 게임 甲
D97656: 적당한 그래픽 적당한 스토리 적당한 몰입감 적당한 적 적당한 버그 적당한 DLC 적당한 게임성으로 적당한 게임이다.


,문서,순위,Count 단어,Count,TF-IDF 단어,TF-IDF,df,idf
0,D79111,1,10,11,10,0.5401,698,5.9633
1,D79111,2,8,1,예술작품이,0.3151,1,11.8198
2,D79111,3,게임성,1,인생작,0.3151,1,11.8198
3,D79111,4,게임을,1,되다,0.2632,13,9.8739
4,D79111,5,그래픽,1,넘어선,0.2614,14,9.8049
5,D19400,1,흔한,9,흔한,0.5759,82,8.0941
6,D19400,2,게임,2,좀비슈터,0.2630,1,11.8198
7,D19400,3,그래픽,1,좀비슈터가,0.2630,1,11.8198
8,D19400,4,노잼,1,학살잼,0.2630,1,11.8198
9,D19400,5,단순,1,좀비학살,0.2476,3,11.1266


### 차이가 생긴 이유

임의의 문서에 단어 $w$가 나타날 확률을 $p(w)=df(w)/N$으로 보면, 그 사건의 자기정보량은 $I(w)=-\ln p(w)=\ln(N/df(w))$이다. 흔한 단어는 관측해도 놀랍지 않아 정보량이 작고, 적은 문서에만 나온 단어는 관측의 놀라움이 커서 정보량이 크다. scikit-learn은 0 나눗셈을 막고 모든 단어의 가중치가 0이 되는 것을 피하려고 $\ln((1+N)/(1+df))+1$을 쓴다. 따라서 Count에서는 반복어가 상위에 남지만, TF-IDF에서는 반복 횟수가 같더라도 문서 집합에서 드문 표현이 위로 올라간다. 이 과정에서 TF-IDF는 단어 순서와 문맥을 버리고, 문서를 구별하지 못하는 공통어의 영향도 줄인다.

**AI 활용 기록**

- Count와 TF-IDF 상위어 차이
- idf의 자기정보량 해석

## 2. BM25 파라미터 실험

Count와 TF-IDF에서 사용한 정규식 토큰화를 그대로 사용했다. BM25 idf는 음수를 피하는 $\ln(1+(N-df+0.5)/(df+0.5))$를 썼다.

In [5]:
import math

N = len(docs)
lengths = np.asarray(X_count.sum(axis=1)).ravel().astype(float)
avgdl = lengths.mean()

def bm25(query, k1=1.5, b=0.75):
    scores = np.zeros(N, dtype=float)
    for word in query.split():
        j = count_vec.vocabulary_.get(word)
        if j is None:
            continue
        f = X_count[:, j].toarray().ravel()
        word_idf = math.log(1 + (N - df[j] + 0.5) / (df[j] + 0.5))
        denom = f + k1 * (1 - b + b * lengths / avgdl)
        scores += word_idf * f * (k1 + 1) / np.where(denom == 0, 1, denom)
    return scores

queries = ["스토리 그래픽", "버그 최적화", "멀티플레이 친구"]
settings = [(0.8, 0.0), (1.2, 0.75), (2.0, 0.75), (2.0, 1.0), (100.0, 0.75)]
bm25_rows, mean_rows = [], []

for query in queries:
    for k1, b in settings:
        scores = bm25(query, k1, b)
        top = np.lexsort((np.arange(N), -scores))[:5]
        mean_rows.append({
            "질의": query,
            "k1": k1,
            "b": b,
            "상위 5개 문서": ", ".join(f"D{i + 1}" for i in top),
            "평균 길이": round(float(lengths[top].mean()), 1),
        })
        for rank, i in enumerate(top, 1):
            bm25_rows.append({
                "질의": query,
                "k1": k1,
                "b": b,
                "순위": rank,
                "문서": f"D{i + 1}",
                "점수": round(float(scores[i]), 3),
                "길이": int(lengths[i]),
                "내용": docs[i][:55],
            })

bm25_summary = pd.DataFrame(mean_rows)
display(bm25_summary)
display(pd.DataFrame(bm25_rows))

,질의,k1,b,상위 5개 문서,평균 길이
0,스토리 그래픽,0.8,0.00,"D13098, D27280, D88991, D1083, D1324",22.0
1,스토리 그래픽,1.2,0.75,"D26388, D89447, D4387, D75892, D85870",5.2
2,스토리 그래픽,2.0,0.75,"D26388, D89447, D4387, D75892, D85870",5.2
3,스토리 그래픽,2.0,1.00,"D26388, D89447, D4387, D75892, D85870",5.2
4,스토리 그래픽,100.0,0.75,"D26388, D89447, D4387, D75892, D85870",5.2
5,버그 최적화,0.8,0.00,"D13704, D34822, D36512, D49817, D56502",21.4
6,버그 최적화,1.2,0.75,"D13704, D96937, D72018, D15320, D34822",9.6
7,버그 최적화,2.0,0.75,"D13704, D96937, D27420, D15320, D395",5.4
8,버그 최적화,2.0,1.00,"D15320, D13704, D395, D496, D2854",3.0
9,버그 최적화,100.0,0.75,"D27420, D31754, D11046, D35444, D92801",8.4


,질의,k1,b,순위,문서,점수,길이,내용
0,스토리 그래픽,0.8,0.00,1,D13098,10.213,28,"음악, 그래픽, 스토리 다 제 마음에 쏙 들었어요 좀 더 액션성과 퍼즐들이 있었으면..."
1,스토리 그래픽,0.8,0.00,2,D27280,10.129,23,GOOD 꽤 괞찮은 브금 그래픽 간단한 컨트롤 BAD 짧은 스토리 영어 크리 무언가...
2,스토리 그래픽,0.8,0.00,3,D88991,10.129,14,스토리 소재와 그래픽 디자인은 좋은데 너무 뻔한 루트에다가 스토리 자체가 너무 흔한...
3,스토리 그래픽,0.8,0.00,4,D1083,8.899,19,"좋은 그래픽, 좋지만 짧은 스토리. 전작과는 다른 전투 시스템을 도입했지만 역시나 ..."
4,스토리 그래픽,0.8,0.00,5,D1324,8.899,26,엄청 정교하고 부드러운 그래픽 + 아주편한 조작감 전작인 림보와 같이 어두운 분위기...
...,...,...,...,...,...,...,...,...
70,멀티플레이 친구,100.0,0.75,1,D43615,26.588,6,1.친구 2.친구 3.친구
71,멀티플레이 친구,100.0,0.75,2,D65326,18.860,1,친구
72,멀티플레이 친구,100.0,0.75,3,D46972,17.981,6,친구 있으면 개꿀잼 친구 없으면 사지마셈
73,멀티플레이 친구,100.0,0.75,4,D50794,16.535,2,멀티플레이 내놔 ^^ㅣ발롬들아


### b와 문서 길이

$b=0$이면 분모에서 문서 길이 항이 사라져 질의어를 여러 번 포함한 긴 리뷰가 상대적으로 유리하다. $b$를 0.75 또는 1로 높이면 평균보다 긴 문서의 분모가 커지고 같은 빈도의 질의어가 있어도 점수가 내려간다. 이 말뭉치에서도 세 질의 모두 $b=0$일 때보다 $b=1$일 때 상위 5개 문서의 평균 길이가 작아졌다. $k_1$을 크게 하면 빈도 포화가 늦어져 반복 횟수의 영향이 다시 커지지만, $b$가 남아 있는 한 길이 보정도 함께 작동한다.

**AI 활용 기록**

- BM25의 $k_1$, $b$ 역할
- 파라미터에 따른 순위와 문서 길이 변화

## 3. Kiwi 토큰화 오류 분류

아래 사례는 말뭉치에서 찾았다. 기대 결과는 문장의 의미와 일반적인 표기를 기준으로 정했다. 각 사례마다 새 `Kiwi` 객체를 만들어 사용자 사전 추가 전후만 비교했다.

In [6]:
from kiwipiepy import Kiwi

error_cases = [
    {"문서": "D128", "문장": "상당히 지루한 오픈월드", "원인": "복합명사", "표제어": "오픈월드", "품사": "NNP", "기대": "오픈월드/NNP"},
    {"문서": "D300", "문장": "한때 인생게임이었습니다", "원인": "복합명사", "표제어": "인생게임", "품사": "NNG", "기대": "인생게임/NNG"},
    {"문서": "D4303", "문장": "메인 스토리라인만 끝내도 충분한편입니다", "원인": "복합명사", "표제어": "스토리라인", "품사": "NNG", "기대": "스토리라인/NNG"},
    {"문서": "D5916", "문장": "시간순삭", "원인": "띄어쓰기", "표제어": "시간순삭", "품사": "NNG", "기대": "시간순삭"},
    {"문서": "D45", "문장": "갓겜 세일할때 꼭 사셈", "원인": "신조어", "표제어": "사셈", "품사": "NNG", "기대": "사/VV + 셈/종결 표현"},
]

token_rows = []
for case in error_cases:
    kiwi = Kiwi()
    before_tokens = " + ".join(f"{t.form}/{t.tag}" for t in kiwi.tokenize(case["문장"]))
    before_space = kiwi.space(case["문장"])
    kiwi.add_user_word(case["표제어"], case["품사"])
    after_tokens = " + ".join(f"{t.form}/{t.tag}" for t in kiwi.tokenize(case["문장"]))
    after_space = kiwi.space(case["문장"])
    fixed = (before_tokens != after_tokens or before_space != after_space) and case["표제어"] != "사셈"
    token_rows.append({
        "문서": case["문서"],
        "원인": case["원인"],
        "문장": case["문장"],
        "기대": case["기대"],
        "기본 분석": before_tokens,
        "기본 띄어쓰기": before_space,
        "사전 추가 뒤": after_tokens,
        "추가 뒤 띄어쓰기": after_space,
        "교정 여부": "예" if fixed else "아니요",
    })

token_error_table = pd.DataFrame(token_rows)
display(token_error_table)
print("원인별 건수")
display(token_error_table.groupby("원인").size().rename("건수").reset_index())

,문서,원인,문장,기대,기본 분석,기본 띄어쓰기,사전 추가 뒤,추가 뒤 띄어쓰기,교정 여부
0,D128,복합명사,상당히 지루한 오픈월드,오픈월드/NNP,상당히/MAG + 지루/XR + 하/XSA + ᆫ/ETM + 오픈/NNG + 월드/NNG,상당히 지루한 오픈 월드,상당히/MAG + 지루하/VA + ᆫ/ETM + 오픈월드/NNP,상당히 지루한 오픈월드,예
1,D300,복합명사,한때 인생게임이었습니다,인생게임/NNG,한때/NNG + 인생/NNG + 게임/NNG + 이/VCP + 었/EP + 습니다/EF,한때 인생 게임이었습니다,한때/NNG + 인생게임/NNG + 이/VCP + 었/EP + 습니다/EF,한때 인생게임이었습니다,예
2,D4303,복합명사,메인 스토리라인만 끝내도 충분한편입니다,스토리라인/NNG,메인/NNG + 스토리/NNG + 라인/NNG + 만/JX + 끝내/VV + 어도/...,메인 스토리 라인만 끝내도 충분한 편입니다,메인/NNG + 스토리라인/NNG + 만/JX + 끝내/VV + 어도/EC + 충분...,메인 스토리라인만 끝내도 충분한 편입니다,예
3,D5916,띄어쓰기,시간순삭,시간순삭,시간순삭/NNG,시간 순 삭,시간순삭/NNG,시간순삭,예
4,D45,신조어,갓겜 세일할때 꼭 사셈,사/VV + 셈/종결 표현,갓겜/NNP + 세일/NNG + 하/XSV + ᆯ/ETM + 때/NNG + 꼭/MA...,갓겜 세일할 때 꼭 사셈,갓겜/NNP + 세일/NNG + 하/XSV + ᆯ/ETM + 때/NNG + 꼭/MA...,갓겜 세일할 때 꼭 사셈,아니요


원인별 건수


,원인,건수
0,띄어쓰기,1
1,복합명사,3
2,신조어,1


### 오류 해석

`오픈월드`, `인생게임`, `스토리라인`은 구성 성분이 각각 사전에 있어 Kiwi가 더 작은 명사로 나눈 사례다. 사용자 단어 등록 뒤 원하는 복합명사 단위가 유지됐다. `시간순삭`은 형태소 분석 결과에서는 한 단어였지만 띄어쓰기 교정기가 `시간 순 삭`으로 과분할했다. 사용자 사전이 이 오류도 막았다. 반면 `사셈`은 동사 `사다`에 온라인 종결 표현이 붙은 형태인데 기본 분석은 전체를 일반명사로 처리했다. 단순 사용자 단어 등록은 표면형의 품사만 지정하므로 내부 형태소 경계를 복원하지 못했다. 이 사례에는 사전 단어 하나보다 사전 분석 항목이나 형태 규칙 보완이 필요하다.

**AI 활용 기록**

- Kiwi의 한국어 분석 오류 원인
- 사용자 사전 적용 방법

## 도전 과제. 보간 Kneser-Ney 트라이그램

문서 경계 표지를 따로 넣지 않고 말뭉치 순서대로 90%를 학습, 10%를 평가에 사용했다. 모든 n-gram 비교에서 같은 정규식 토큰화, 같은 분할, 같은 `<UNK>` 처리를 썼다. 학습에서 한 번만 나온 단어와 평가 시 미등록어를 `<UNK>`로 묶었다. 트라이그램의 하위 바이그램은 원시 빈도가 아니라 서로 다른 왼쪽 문맥 수를 사용했다.

In [7]:
from collections import Counter, defaultdict

analyzer = count_vec.build_analyzer()
all_tokens = [token for doc in docs for token in analyzer(doc)]
cut = int(len(all_tokens) * 0.9)
train_raw, test_raw = all_tokens[:cut], all_tokens[cut:]
raw_freq = Counter(train_raw)
vocabulary = {w for w, c in raw_freq.items() if c >= 2}
vocabulary.add("<UNK>")
train = [w if w in vocabulary else "<UNK>" for w in train_raw]
test = [w if w in vocabulary else "<UNK>" for w in test_raw]

uni = Counter(train)
bi = Counter(zip(train, train[1:]))
tri = Counter(zip(train, train[1:], train[2:]))

bi_followers = defaultdict(set)
preceders = defaultdict(set)
for a, b in bi:
    bi_followers[a].add(b)
    preceders[b].add(a)
n_bigram_types = len(bi)

tri_followers = defaultdict(set)
tri_history_count = Counter()
continuation_bigram = Counter()
for a, b, c in tri:
    tri_followers[(a, b)].add(c)
    tri_history_count[(a, b)] += tri[(a, b, c)]
    continuation_bigram[(b, c)] += 1

continuation_history_count = Counter()
continuation_followers = defaultdict(set)
for (b, c), value in continuation_bigram.items():
    continuation_history_count[b] += value
    continuation_followers[b].add(c)

def p_cont(w):
    return len(preceders[w]) / n_bigram_types

def p_kn_bigram(prev, word, D):
    c_prev = uni[prev]
    if c_prev == 0:
        return p_cont(word)
    discounted = max(bi[(prev, word)] - D, 0) / c_prev
    backoff = D * len(bi_followers[prev]) / c_prev
    return discounted + backoff * p_cont(word)

def p_kn_trigram(a, b, word, D):
    lower_den = continuation_history_count[b]
    if lower_den:
        lower_discounted = max(continuation_bigram[(b, word)] - D, 0) / lower_den
        lower_backoff = D * len(continuation_followers[b]) / lower_den
        lower = lower_discounted + lower_backoff * p_cont(word)
    else:
        lower = p_cont(word)

    history = (a, b)
    upper_den = tri_history_count[history]
    if upper_den == 0:
        return lower
    upper_discounted = max(tri[(a, b, word)] - D, 0) / upper_den
    upper_backoff = D * len(tri_followers[history]) / upper_den
    return upper_discounted + upper_backoff * lower

test_triples = list(zip(test, test[1:], test[2:]))

def perplexity(order, D):
    log_sum = 0.0
    for a, b, w in test_triples:
        p = p_kn_bigram(b, w, D) if order == 2 else p_kn_trigram(a, b, w, D)
        log_sum += math.log2(max(p, 1e-300))
    return 2 ** (-log_sum / len(test_triples))

ppl_rows = []
for D in [0.5, 0.75, 0.9]:
    for order in [2, 3]:
        ppl_rows.append({"n-gram": f"{order}-gram", "D": D, "PPL": perplexity(order, D)})

ppl_table = pd.DataFrame(ppl_rows)
ppl_pivot = ppl_table.pivot(index="D", columns="n-gram", values="PPL").reset_index()
ppl_pivot["3-gram 개선율(%)"] = (
    (ppl_pivot["2-gram"] - ppl_pivot["3-gram"]) / ppl_pivot["2-gram"] * 100
)
print(f"전체 토큰 {len(all_tokens):,}, 학습 {len(train):,}, 평가 {len(test):,}, 어휘 {len(vocabulary):,}")
display(ppl_table.assign(PPL=lambda x: x.PPL.round(2)))
display(ppl_pivot.round(2))

전체 토큰 1,062,592, 학습 956,332, 평가 106,260, 어휘 61,515


,n-gram,D,PPL
0,2-gram,0.50,1327.39
1,3-gram,0.50,1680.75
2,2-gram,0.75,1166.75
3,3-gram,0.75,1270.19
4,2-gram,0.90,1138.45
5,3-gram,0.90,1166.02


n-gram,D,2-gram,3-gram,3-gram 개선율(%)
0,0.50,1327.39,1680.75,-26.62
1,0.75,1166.75,1270.19,-8.87
2,0.90,1138.45,1166.02,-2.42


### 퍼플렉서티 해석

바이그램과 트라이그램은 바로 위 표처럼 같은 평가 위치에서 비교했다. 트라이그램은 직전 두 단어가 만드는 더 구체적인 문맥을 사용하고, 보지 못한 문맥에서는 continuation count로 만든 하위 바이그램으로 보간한다. 할인 $D$가 커질수록 관측 n-gram에서 더 많은 확률 질량을 떼어 하위 모델에 넘긴다. 따라서 너무 작은 $D$는 관측 조합을 과신하고, 너무 큰 $D$는 유용한 관측 빈도까지 약화시킬 수 있다. 이 말뭉치에 가장 맞는 값은 세 후보의 실제 PPL 최솟값으로 판단한다.

**AI 활용 기록**

- 보간 Kneser-Ney의 트라이그램 확장
- 할인값 $D$에 따른 퍼플렉서티 변화